In [5]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "12345678"
auth=(username, password)
driver = GraphDatabase.driver(uri, auth=(username, password))

In [6]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Neo4jMapReduce") \
    .master("local[*]") \
    .getOrCreate()

In [7]:
def Map(driver):

    query = """
    MATCH (c)
    WHERE c.kind = "Compound"
    OPTIONAL MATCH (c)-[r]->()
    RETURN c.name AS Compound, r.metaedge AS metaedge
    """
    with driver.session() as session:
        result = [dict(record) for record in session.run(query)]

    rdd = spark.sparkContext.parallelize(result)
    pairs = rdd.map(lambda x: (x['Compound'], x['metaedge']))
    return pairs

In [8]:
Map(driver)

PythonRDD[3] at RDD at PythonRDD.scala:53

In [9]:
def Sort(pairs):

    gene_types = ['CbG', 'CuG', 'CdG']
    genes = pairs.filter(lambda x: x[1] in gene_types).map(lambda x: (x[0], 1))

    disease_types = ['CtD', 'CpD']
    diseases = pairs.filter(lambda x: x[1] in disease_types).map(lambda x: (x[0], 1))
    
    return genes, diseases


In [17]:
def Reduce(genes, diseases):


    gene_counts = genes.reduceByKey(lambda a, b: a + b)
    desc = gene_counts.sortBy(lambda x: x[1], ascending=False)

    disease_counts = diseases.reduceByKey(lambda a, b: a + b)
    disease = disease_counts.collect()

    final = desc.take(5)
    
    return final, disease
   


In [18]:
def MapReduce(driver):
    pairs = Map(driver)
    genes, diseases = Sort(pairs)

    result, disease_list = Reduce(genes, diseases)
    disease_dict = dict(disease_list)

    for compound, gene_count in result:
        disease_count = disease_dict.get(compound, 0)
        print(f"Compound: {compound}, Gene Count: {gene_count}, Disease Count: {disease_count}")

In [19]:
result = MapReduce(driver)

Compound: Crizotinib, Gene Count: 585, Disease Count: 1
Compound: Dasatinib, Gene Count: 564, Disease Count: 1
Compound: Doxorubicin, Gene Count: 532, Disease Count: 17
Compound: Vinblastine, Gene Count: 523, Disease Count: 7
Compound: Digoxin, Gene Count: 522, Disease Count: 2


In [ ]:
spark.stop()
driver.close()